In [ ]:
# !pip install langchain
# !pip install langchain-community
# !pip install -U langchain-google-genai
# !pip install --upgrade langchain-core langchain-google-vertexai
# !pip install --upgrade langchain-community duckduckgo-search
# !pip install langgraph
# !pip install requests pubchempy rdkit-pypi
# !pip install ctx-python
# !pip install biopython
# !pip install langgraph-checkpoint-sqlite
# !pip uninstall -y faiss-cpu
# !pip install langchain_openai python-dotenv
# !pip install "numpy<2"  # Use NumPy 1.x for compatibility with rdkit-pypi
# !pip install beautifulsoup4

In [2]:
# export LANGSMITH_TRACING="true"
# export LANGSMITH_API_KEY="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"
# export LANGSMITH_PROJECT="default" # or any other project name
# openai_api_key = "REMOVED_CREDENTIAL"

In [1]:
import os
from typing import Optional

from dotenv import load_dotenv
from langchain_core.utils.utils import secret_from_env
from langchain_openai import ChatOpenAI
from pydantic import Field, SecretStr

load_dotenv()

class ChatOpenRouter(ChatOpenAI):
    openai_api_key: Optional[SecretStr] = Field(
        alias="api_key", default_factory=secret_from_env("OPENROUTER_API_KEY", default=None)
    )
    @property
    def lc_secrets(self) -> dict[str, str]:
        return {"openai_api_key": "OPENROUTER_API_KEY"}

    def __init__(self,
                 openai_api_key: Optional[str] = None,
                 **kwargs):
        openai_api_key = openai_api_key or os.environ.get("OPENROUTER_API_KEY")
        super().__init__(base_url="https://openrouter.ai/api/v1", openai_api_key=openai_api_key, **kwargs)


# model = ChatOpenRouter(model_name="google/gemini-2.0-flash-001")

In [ ]:
import sys, pathlib, os
# Ensure local 'code' folder is importable
sys.path.append('code')

# Import the modularized tools
from compound_tool import CompoundDatabaseTool
from searching_tool import FullTextJournalSearchTool

print("Initialized tools from code/ (compound_tool, searching_tool)")

# Quick compound tool smoke test
comp_tool = CompoundDatabaseTool()
print('\nCompound tool test: aspirin')
try:
    out = comp_tool._run(compound_name='aspirin')
    print(out if isinstance(out, str) else str(out)[:1000])
except Exception as e:
    print('Compound test failed:', e)

# Quick journal tool smoke test (will create a run under searching_results/)
journal_tool = FullTextJournalSearchTool()
print('\nJournal tool test: running a short search (max_results=2)')
try:
    saved_files = journal_tool._run(query='machine learning', max_results=2)
    print('Saved files count:', len(saved_files))
    if saved_files:
        print('First saved file:', saved_files[0])
except Exception as e:
    print('Journal test failed:', e)


2025-10-05 19:52:56,292 | INFO | Enabling RDKit 2022.09.5 jupyter extensions


In [1]:
print("\\nTesting Journal Search Tool...")
journal_tool = FullTextJournalSearchTool()

# Test 4: Journal search
print("\\n4. Testing journal search (limited to 2 results):")
import os
print(f"Current working directory: {os.getcwd()}")
result4 = journal_tool._run(query="machine learning drug discovery", max_results=20)
print("Type of result4:", type(result4))
print("✓ Journal search completed" if result4 else "✗ Journal search failed")
if isinstance(result4, list):
    print(f"Result4 is a list with {len(result4)} items")
    if result4:
        print("First item keys:", list(result4[0].keys()) if isinstance(result4[0], dict) else "Not a dict")
    # Check saved files
    if os.path.exists("searching_results"):
        files = []
        for root, dirs, filenames in os.walk("searching_results"):
            for filename in filenames:
                if filename.endswith('.json'):
                    files.append(os.path.join(root, filename))
        print(f"JSON files found: {len(files)}")
        if files:
            print("Sample file:", files[0])
    else:
        print("searching_results folder not found")

# Test 5: Test article matching function with more similar titles
print("\\n5. Testing article matching function:")
test_title1 = "Machine Learning in Drug Discovery"
test_title2 = "Machine Learning for Drug Discovery Applications"
match_result = journal_tool.is_same_article(test_title1, test_title2)
print(f"✓ Article matching: {match_result} (should be True for similar titles)")

print("\\n" + "="*50)
print("TEST SUMMARY:")

\nTesting Journal Search Tool...


NameError: name 'FullTextJournalSearchTool' is not defined